# Tarefa 3 — Aplicação: Montagem de Panorama (Image Stitching)
**Atividade 3 · TECA2 20261 · Pontos de Interesse e seus Descritores**

**Alunos:** Henryque Oliveira, Matheus Marinho e Rodrigo Oliveira

Este notebook implementa o pipeline completo de image stitching: detectar keypoints → extrair descritores → corresponder → estimar homografia com RANSAC → aplicar warp perspectivo → compor panorama.

### Base teórica

- **Homografia**: transformação projetiva 3×3 que mapeia pontos de um plano de imagem para outro, preservando linhas retas. Dada uma correspondência entre dois conjuntos de pontos, `cv2.findHomography` estima a matriz H tal que `p2 ≈ H · p1` (coordenadas homogêneas). Para montar o panorama, usamos H para reprojetar (`warp`) img1 no sistema de coordenadas de img2.
- **RANSAC**: estima H de forma robusta sorteando subconjuntos mínimos de 4 pares de pontos, ajustando H, contando inliers (pares cuja distância reproj. < threshold) e guardando o melhor modelo. Outliers — matches incorretos que passaram pelo Ratio Test — são automaticamente descartados.
- **USAC_MAGSAC**: variante moderna disponível em OpenCV ≥ 4.5 como `cv2.USAC_MAGSAC` dentro de `cv2.findHomography`. Mais robusto que o RANSAC clássico, especialmente quando a proporção de outliers é alta. O restante do código permanece idêntico.

> Referências: Corke (2023) Seções 14.1–14.2.4 · Brown & Lowe (2007) AutoStitch · Slides TECA2 20261, slides 15–18.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print('OpenCV:', cv2.__version__)
print('SIFT disponível:', hasattr(cv2, 'SIFT_create'))
print('AKAZE disponível:', hasattr(cv2, 'AKAZE_create'))

PATH_ESQ = 'images/mesa_esq.jpeg'
PATH_DIR = 'images/mesa_dir.jpeg'

def load(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img, gray

OpenCV: 4.13.0
SIFT disponível: True
AKAZE disponível: True
